Name: Lin Qizhou

Key insights / takeaways:
- I trained a tabular generative model (CTGAN) on the fraud transactions dataset and generated a 5,000-row synthetic dataset while preserving the original schema.
- I evaluated synthetic data quality using numeric distribution similarity (KS statistic), categorical distribution similarity (TVD), correlation structure difference, and a downstream utility check when a label column is available.
- I summarized practical trade-offs observed during fine-tuning, including stability, mode collapse risks, and how target/label scarcity affects utility metrics.

In [1]:
!pip -q uninstall -y sdv rdt copulas ctgan pyarrow pandas numpy scikit-learn
!pip -q install -U --no-cache-dir "numpy==1.26.4" "pandas==2.2.2" "pyarrow==16.1.0" "scikit-learn==1.5.2"
!pip -q install -U --no-cache-dir "sdv==1.17.1"
print("done")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 94.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 170.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 82.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sdmetrics 0.27.2 requires copulas>=0.12.1; python_version < "3.14", which is not installed.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, bu

In [2]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded.keys()))

Saving fraud_transactions.csv to fraud_transactions (1).csv
['fraud_transactions (1).csv']


In [3]:
from pathlib import Path
import pandas as pd

csvs = sorted(Path("/content").glob("*.csv"))
if not csvs:
    raise FileNotFoundError("no csv under /content")

csv_path = [p for p in csvs if "fraud" in p.name.lower()]
csv_path = str(csv_path[0] if csv_path else csvs[0])

df = pd.read_csv(csv_path)
print(csv_path)
print(df.shape)
df.head()

/content/fraud_transactions (1).csv
(6486, 8)


,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,2/27/19 21:32,"fraud_Langosh, Wintheiser and Hyatt",food_dining,83.64,F,TX,"Physicist, medical",0
1,2/13/19 19:41,fraud_Dibbert and Sons,entertainment,79.13,M,PA,Secretary/administrator,0
2,1/11/19 20:03,"fraud_McDermott, Osinski and Morar",home,12.02,F,CA,"Buyer, industrial",0
3,1/20/19 9:08,fraud_Bauch-Raynor,grocery_pos,84.41,M,TN,Clothing/textile technologist,0
4,1/4/19 17:04,"fraud_Reichert, Huels and Hoppe",shopping_net,2.81,F,ME,Financial trader,0


In [4]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=df)

synth = CTGANSynthesizer(metadata=metadata, epochs=300, verbose=True)
synth.fit(df)

synthetic = synth.sample(num_rows=5000)
print(synthetic.shape)
synthetic.head()

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:120: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:105: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x7eb4b41f1da0>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath

PerformanceAlert: Using the CTGANSynthesizer on this data is not recommended. To model this data, CTGAN will generate a large number of columns.

Original Column Name   Est # of Columns (CTGAN)
merchant               692
category               14
amt                    11
gender                 2
job                    472
is_fraud               2

We recommend preprocessing discrete columns that can have many values, using 'update_transformers'. Or you may drop columns that are not necessary to model. (Exit this script using ctrl-C)


Gen. (-00.34) | Discrim. (-00.01): 100%|██████████| 300/300 [50:34<00:00, 10.11s/it]


(5000, 8)


,trans_date_trans_time,merchant,category,amt,gender,state,job,is_fraud
0,sdv-pii-vh814,fraud_Friesen Inc,grocery_pos,59.27,M,Kansas,Research scientist (physical sciences),0
1,sdv-pii-hp7b3,"fraud_Wuckert, Wintheiser and Friesen",home,155.73,F,Louisiana,"Engineer, aeronautical",0
2,sdv-pii-csend,fraud_Hilpert-Conroy,gas_transport,94.22,M,New Hampshire,"Horticulturist, commercial",0
3,sdv-pii-3ozf4,"fraud_Langworth, Boehm and Gulgowski",gas_transport,116.02,F,Maryland,Product designer,0
4,sdv-pii-f444r,fraud_Nader-Heller,grocery_pos,186.24,M,Ohio,Art therapist,0


In [5]:
out_path = "/content/fraud_transactions_synthetic_5000.csv"
synthetic.to_csv(out_path, index=False)
print(out_path)

/content/fraud_transactions_synthetic_5000.csv


In [6]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

real = df.copy()
syn = synthetic.copy()

common_cols = [c for c in real.columns if c in syn.columns]
real = real[common_cols]
syn = syn[common_cols]

num_cols = [c for c in common_cols if pd.api.types.is_numeric_dtype(real[c])]
cat_cols = [c for c in common_cols if c not in num_cols]

def ks_stat(a, b):
    a = pd.Series(a).dropna().to_numpy()
    b = pd.Series(b).dropna().to_numpy()
    if len(a) == 0 or len(b) == 0:
        return np.nan
    a = np.sort(a)
    b = np.sort(b)
    allv = np.sort(np.unique(np.concatenate([a, b])))
    cdf_a = np.searchsorted(a, allv, side="right") / len(a)
    cdf_b = np.searchsorted(b, allv, side="right") / len(b)
    return float(np.max(np.abs(cdf_a - cdf_b)))

def tvd_cat(a, b):
    pa = a.fillna("__nan__").value_counts(normalize=True)
    pb = b.fillna("__nan__").value_counts(normalize=True)
    idx = pa.index.union(pb.index)
    pa = pa.reindex(idx, fill_value=0.0)
    pb = pb.reindex(idx, fill_value=0.0)
    return float(0.5 * np.abs(pa - pb).sum())

ks_scores = {c: ks_stat(real[c], syn[c]) for c in num_cols}
tvd_scores = {c: tvd_cat(real[c], syn[c]) for c in cat_cols}

corr_real = real[num_cols].corr().fillna(0.0) if len(num_cols) else pd.DataFrame()
corr_syn = syn[num_cols].corr().fillna(0.0) if len(num_cols) else pd.DataFrame()
corr_diff = float(np.mean(np.abs((corr_real - corr_syn).to_numpy()))) if len(num_cols) else np.nan

target_col = None
for cand in ["is_fraud", "fraud", "label", "target", "Class"]:
    if cand in real.columns:
        target_col = cand
        break

utility_auc = np.nan
if target_col is not None and pd.api.types.is_numeric_dtype(real[target_col]):
    Xr = real.drop(columns=[target_col])
    yr = real[target_col]
    Xs = syn.drop(columns=[target_col])
    ys = syn[target_col]

    Xr_train, Xr_test, yr_train, yr_test = train_test_split(
        Xr, yr, test_size=0.3, random_state=42,
        stratify=yr if yr.nunique() == 2 else None
    )

    pre = ColumnTransformer(
        transformers=[
            ("num", "passthrough", [c for c in Xr.columns if pd.api.types.is_numeric_dtype(Xr[c])]),
            ("cat", OneHotEncoder(handle_unknown="ignore"), [c for c in Xr.columns if not pd.api.types.is_numeric_dtype(Xr[c])]),
        ]
    )

    clf = Pipeline(steps=[("pre", pre), ("lr", LogisticRegression(max_iter=500))])
    clf.fit(Xs, ys)

    if hasattr(clf, "predict_proba") and yr_test.nunique() == 2:
        proba = clf.predict_proba(Xr_test)[:, 1]
        utility_auc = float(roc_auc_score(yr_test, proba))

summary = pd.DataFrame(
    [
        {"metric": "n_rows_real", "value": int(len(real))},
        {"metric": "n_rows_synth", "value": int(len(syn))},
        {"metric": "mean_ks_numeric", "value": float(np.nanmean(list(ks_scores.values()))) if ks_scores else np.nan},
        {"metric": "mean_tvd_categorical", "value": float(np.nanmean(list(tvd_scores.values()))) if tvd_scores else np.nan},
        {"metric": "mean_abs_corr_diff_numeric", "value": corr_diff},
        {"metric": "downstream_auc_train_on_synth_test_on_real", "value": utility_auc},
    ]
)

print(summary)

                                       metric        value
0                                 n_rows_real  6486.000000
1                                n_rows_synth  5000.000000
2                             mean_ks_numeric     0.106585
3                        mean_tvd_categorical     0.440875
4                  mean_abs_corr_diff_numeric     0.033483
5  downstream_auc_train_on_synth_test_on_real     0.865731


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [7]:
from google.colab import files
files.download("/content/fraud_transactions_synthetic_5000.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>